In [1]:
import torch
import torch.nn as nn
from torchvision import models
from transformers import AlbertModel, AlbertTokenizer
from PIL import Image
from torchvision import transforms
# Define CBAM for Image Feature Extraction
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction_ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction_ratio, kernel_size=1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_channels // reduction_ratio, in_channels, kernel_size=1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.mlp(self.avg_pool(x))
        max_out = self.mlp(self.max_pool(x))
        out = avg_out + max_out
        return self.sigmoid(out) * x

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x = torch.cat([avg_out, max_out], dim=1)
        x = self.conv(x)
        return self.sigmoid(x)

class CBAM(nn.Module):
    def __init__(self, in_channels, reduction_ratio=16, kernel_size=7):
        super(CBAM, self).__init__()
        self.channel_attention = ChannelAttention(in_channels, reduction_ratio)
        self.spatial_attention = SpatialAttention(kernel_size)

    def forward(self, x):
        x = self.channel_attention(x)
        x = self.spatial_attention(x) * x
        return x



class DenseNetCBAMFeatureExtractor(nn.Module):
    def __init__(self):
        super(DenseNetCBAMFeatureExtractor, self).__init__()
        self.densenet = models.densenet121(pretrained=True)
        self.densenet.classifier = nn.Identity()
        self.cbam = CBAM(in_channels=1024)
        self.fc_reduction = nn.Linear(1024, 512)  # Change output dimension to 512

    def forward(self, x):
        features = self.densenet.features(x)
        refined_features = self.cbam(features)
        feature_vector = torch.mean(refined_features, dim=[2, 3])
        reduced_vector = self.fc_reduction(feature_vector)
        return reduced_vector


class ALBERTBiLSTMFeatureExtractor(nn.Module):
    def __init__(self, embedding_dim=512, lstm_hidden_size=256):
        super(ALBERTBiLSTMFeatureExtractor, self).__init__()
        self.albert = AlbertModel.from_pretrained("albert-base-v2", output_hidden_states=True)
        self.tokenizer = AlbertTokenizer.from_pretrained("albert-base-v2")
        self.fc_projection = nn.Linear(768, embedding_dim)
        self.bilstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=lstm_hidden_size,
            num_layers=1,
            bidirectional=True,
            batch_first=True
        )

    def forward(self, input_text):
        inputs = self.tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=150)
        albert_outputs = self.albert(**inputs)
        contextual_embeddings = albert_outputs.last_hidden_state
        projected_embeddings = self.fc_projection(contextual_embeddings)
        bilstm_outputs, _ = self.bilstm(projected_embeddings)
        text_feature_vector = torch.mean(bilstm_outputs, dim=1)  # Global average pooling
        return text_feature_vector


In [2]:
# Image preprocessing function
def preprocess_image(image_path):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),  # Resize to the input size required by DenseNet
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet normalization
    ])
    image = Image.open(image_path).convert("RGB")
    return transform(image).unsqueeze(0)  # Add batch dimension

# Path to the image
image_path = '/kaggle/input/mvsasingle/MVSA_Single/data/1.jpg'

# Load and preprocess the image
input_image = preprocess_image(image_path)

# Instantiate the feature extraction model
model = DenseNetCBAMFeatureExtractor()


model.eval()
with torch.no_grad():
    feature_vector = model(input_image)
    print("Image Feature Vector:", feature_vector)
    print("Feature Vector Shape:", feature_vector.shape)

/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth
100%|██████████| 30.8M/30.8M [00:00<00:00, 194MB/s]


Image Feature Vector: tensor([[-1.3875e-01,  3.3327e-02, -1.1985e-01, -1.4785e-01, -9.2312e-03,
          2.7611e-02, -2.5474e-01,  1.2382e-01,  1.8379e-01,  1.8326e-01,
          1.5602e-02,  6.4244e-02,  4.1787e-02,  2.4562e-02, -2.6003e-02,
          4.4064e-02,  5.9618e-02, -8.0310e-02,  1.7969e-01,  1.7242e-01,
         -1.3718e-01, -5.3169e-02, -1.9030e-02, -1.0725e-01,  3.2361e-02,
          5.3968e-02, -3.4440e-01, -2.1556e-02,  1.1688e-01, -1.3372e-01,
          1.6111e-03, -1.4986e-01,  1.1888e-01,  3.7999e-02, -4.7157e-02,
         -6.4413e-02,  8.8837e-02, -1.6253e-01, -1.2375e-02, -2.0365e-01,
          3.3568e-01, -9.2586e-02, -3.0382e-02, -2.3147e-01, -3.3683e-01,
          2.1196e-03,  1.5096e-01,  9.2376e-04,  1.8584e-01,  1.6110e-01,
          1.2563e-01, -1.2656e-01,  2.2167e-01, -1.3382e-02, -8.1190e-02,
         -1.5376e-01,  1.0269e-02, -3.9528e-02,  8.2838e-02, -9.0446e-03,
         -1.3241e-02, -1.3183e-02,  2.5039e-01,  6.9217e-02, -3.9472e-02,
         -7.8508

In [31]:
# Function to read the text from the file
def read_text_file(file_path):
    encodings = ['utf-8', 'ISO-8859-1', 'utf-16']
    for encoding in encodings:
        try:
            with open(file_path, 'r', encoding=encoding) as file:
                text = file.read().strip()
            return text  # Return the text if reading is successful
        except UnicodeDecodeError:
            continue  # Try the next encoding if a Unicode error occurs
    print(f"Skipping file {file_path} due to encoding issues.")
    return None  # Return None if all encodings fail




text_path = '/kaggle/input/mvsasingle/MVSA_Single/data/2081.txt'


input_text = read_text_file(text_path)
print("Input Text:", input_text)

# Instantiate the model
model = ALBERTBiLSTMFeatureExtractor()

# Extract text features
model.eval()
with torch.no_grad():
    text_features = model([input_text]) 
    print("Text Feature Vector:", text_features)
    print("Feature Vector Shape:", text_features.shape)

Input Text: RT @Leighgriff09: Delighted to be back in the team today, another goal, and 3 points! Perfect Valentine's Day! Inter on Thursday now ? http¡­


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/47.4M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/760k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

/opt/conda/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Text Feature Vector: tensor([[-2.5259e-01,  3.9120e-02, -9.9982e-02, -5.7521e-02, -9.5765e-02,
         -9.2023e-02, -2.7530e-02, -9.3075e-03,  2.8381e-01,  3.9494e-02,
          6.3338e-02, -6.7683e-02, -5.4177e-02,  5.5102e-02,  9.8373e-02,
          8.1161e-02,  1.2028e-01, -1.3186e-01, -1.3081e-01,  2.3681e-01,
         -3.3120e-02, -1.5391e-01,  1.5600e-01, -5.7210e-02,  1.8891e-01,
          2.0405e-01,  1.6196e-02,  1.6215e-01, -2.3681e-01,  2.7588e-01,
          1.8950e-01,  3.8644e-02, -1.5231e-01, -2.3234e-01,  1.2553e-01,
         -2.2966e-01,  6.4829e-03,  1.0720e-02, -2.3303e-01, -9.3327e-02,
          2.3037e-01,  2.1703e-01,  3.4971e-01,  6.1825e-02, -1.2490e-01,
          9.3344e-02, -1.4477e-01, -3.2223e-02, -1.2943e-01,  8.1763e-02,
         -1.0175e-01,  1.2954e-01,  2.4067e-01, -1.2959e-01, -1.6460e-01,
          7.9776e-02,  4.0279e-02,  1.6899e-01, -1.0310e-01,  1.1180e-01,
          1.1967e-01,  7.5445e-02,  3.0604e-02, -1.7224e-01, -8.1021e-02,
          4.4135e

In [32]:
import os
import torch
from PIL import Image


data_folder = "/kaggle/input/mvsasingle/MVSA_Single/data"
os.makedirs("features", exist_ok=True)


image_model = DenseNetCBAMFeatureExtractor().eval()  # Image feature extraction model
text_model = ALBERTBiLSTMFeatureExtractor().eval()    # Text feature extraction model


image_model.eval()
text_model.eval()

# Dictionary to store all features
all_features_512 = {}

# Extract and store features in a dictionary, with error handling for images
def extract_features(image_path, text_path, index):
    try:
        # Load and preprocess image
        input_image = preprocess_image(image_path)
        
        # Extract image features
        with torch.no_grad():
            image_features = image_model(input_image)
    except Exception as e:
        print(f"Skipping {image_path} due to an error: {e}")
        return  # Skip this sample if there's an issue with the image
    
    # Read and process text
    input_text = read_text_file(text_path)
    
    # Extract text features
    with torch.no_grad():
        text_features = text_model([input_text])
    
    # Add features to the dictionary
    all_features_512[index] = {
        "image_features": image_features,
        "text_features": text_features
    }

# Loop through all files in the data folder
count = 0
for filename in os.listdir(data_folder):
    if filename.endswith(".jpg"):
        base_filename = os.path.splitext(filename)[0]
        
        
        image_path = os.path.join(data_folder, f"{base_filename}.jpg")
        text_path = os.path.join(data_folder, f"{base_filename}.txt")
        
        # Ensure both files exist
        if os.path.exists(image_path) and os.path.exists(text_path):
            extract_features(image_path, text_path, base_filename)
            count += 1
            
            
            if count % 200 == 0:
                print(f"Processed {count} pairs")


torch.save(all_features_512, "features/all_features_512.pt")
print("All features have been saved to features/all_features_512.pt")

Processed 200 pairs
Processed 400 pairs
Processed 600 pairs
Processed 800 pairs
Processed 1000 pairs
Processed 1200 pairs
Processed 1400 pairs
Processed 1600 pairs
Processed 1800 pairs
Processed 2000 pairs
Processed 2200 pairs
Processed 2400 pairs
Processed 2600 pairs
Processed 2800 pairs
Processed 3000 pairs
Processed 3200 pairs
Processed 3400 pairs
Processed 3600 pairs
Processed 3800 pairs
Processed 4000 pairs
Processed 4200 pairs
Processed 4400 pairs
Processed 4600 pairs
Processed 4800 pairs
All features have been saved to features/all_features_512.pt


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split
from sklearn.metrics import accuracy_score, precision_score, confusion_matrix
from sklearn.metrics import f1_score
import torch.nn.functional as F
import pandas as pd

# Load the combined 512-dimensional features
feature_path = "/kaggle/input/all-features-512/all_features_512.pt"  
all_features_512 = torch.load(feature_path)

# Load labels
label_path = "/kaggle/input/mvsasingle/MVSA_Single/labelResultAll.txt"
sentiment_map = {'neutral': 0, 'positive': 1, 'negative': 2}
data = []

# Load and process the label file
with open(label_path, 'r') as file:
    for line in file:
        parts = line.strip().split('\t')
        if len(parts) == 2:
            sample_id, sentiment = parts
            text_sentiment, image_sentiment = sentiment.split(',')
            data.append([sample_id, text_sentiment, image_sentiment])

labels_df = pd.DataFrame(data, columns=["ID", "text", "image"])


if labels_df.iloc[0]["ID"] == "ID":
    labels_df = labels_df.drop(0).reset_index(drop=True)

valid_ids = [idx for idx in labels_df["ID"] if idx in all_features_512]
labels_df = labels_df[labels_df["ID"].isin(valid_ids)].reset_index(drop=True)


labels_df["text"] = labels_df["text"].map(sentiment_map)
labels_df["image"] = labels_df["image"].map(sentiment_map)
labels_df["combined"] = labels_df.apply(lambda row: max(row["text"], row["image"]), axis=1)


# Define dataset for loading 512-dimensional features
class CombinedFeatureDataset(Dataset):
    def __init__(self, features, labels_df):
        self.features = features
        self.labels_df = labels_df
        
    def __len__(self):
        return len(self.labels_df)
    
    def __getitem__(self, idx):
        sample_id = str(self.labels_df.loc[idx, "ID"])
        feature_data = self.features[sample_id]
        image_features = feature_data["image_features"]
        text_features = feature_data["text_features"]
        combined_label = self.labels_df.loc[idx, "combined"]
        return image_features, text_features, combined_label

# Initialize dataset and data loaders
dataset = CombinedFeatureDataset(all_features_512, labels_df)
train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size
train_data, val_data, test_data = random_split(dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)



In [ ]:

import torch.nn.functional as F

# Cross-Attention Network with Gating
class GatedCrossAttentionNetwork(nn.Module):
    def __init__(self, feature_dim=512, hidden_dim=256, output_dim=3, dropout_rate=0.3):
        super(GatedCrossAttentionNetwork, self).__init__()

        # Linear layers for cross-attention
        self.W_q = nn.Linear(feature_dim, hidden_dim)
        self.W_k = nn.Linear(feature_dim, hidden_dim)
        self.W_v = nn.Linear(feature_dim, hidden_dim)

        # Cross-Interaction Layers
        self.cross_interaction_text = nn.Linear(hidden_dim * 2, hidden_dim)
        self.cross_interaction_image = nn.Linear(hidden_dim * 2, hidden_dim)

        # Gating Mechanism
        self.gate_text = nn.Linear(hidden_dim * 2, hidden_dim)
        self.gate_image = nn.Linear(hidden_dim * 2, hidden_dim)

        # Fusion Layer
        self.fc_fusion = nn.Linear(hidden_dim * 2, hidden_dim)
        self.dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(hidden_dim, output_dim)

    def forward(self, text_features, image_features):
        """ 
        text_features: H_T (unimodal text)
        image_features: H_I (unimodal image)
        """

        # Step 1: Apply Cross-Attention (Text attends to Image)
        Q = self.W_q(text_features)  # Query from text
        K = self.W_k(image_features)  # Key from image
        V = self.W_v(image_features)  # Value from image

        attention_scores = torch.matmul(Q, K.transpose(-2, -1)) / torch.sqrt(torch.tensor(Q.size(-1), dtype=torch.float32))
        attention_weights = F.softmax(attention_scores, dim=-1)
        attended_text = torch.matmul(attention_weights, V)  # C_IT (text attended to image)

        # Step 2: Apply Cross-Attention (Image attends to Text)
        Q_img = self.W_q(image_features)  # Query from image
        K_txt = self.W_k(text_features)  # Key from text
        V_txt = self.W_v(text_features)  # Value from text

        attention_scores_img = torch.matmul(Q_img, K_txt.transpose(-2, -1)) / torch.sqrt(torch.tensor(Q_img.size(-1), dtype=torch.float32))
        attention_weights_img = F.softmax(attention_scores_img, dim=-1)
        attended_image = torch.matmul(attention_weights_img, V_txt)  # C_TI (image attended to text)

        # Step 3: Cross Interaction Layer
        cross_text = torch.tanh(self.cross_interaction_text(torch.cat([attended_text, text_features], dim=-1)))
        cross_image = torch.tanh(self.cross_interaction_image(torch.cat([attended_image, image_features], dim=-1)))

        # Step 4: Gating Mechanism
        gate_text = torch.sigmoid(self.gate_text(torch.cat([cross_text, text_features], dim=-1)))
        gate_image = torch.sigmoid(self.gate_image(torch.cat([cross_image, image_features], dim=-1)))

        final_text_vector = gate_text * cross_text + (1 - gate_text) * text_features
        final_image_vector = gate_image * cross_image + (1 - gate_image) * image_features

        # Step 5: Fusion (Final Fused Vector)
        final_fused_vector = self.fc_fusion(torch.cat([final_text_vector, final_image_vector], dim=-1))
        final_fused_vector = self.dropout(final_fused_vector)

        # Step 6: Classification
        logits = self.classifier(final_fused_vector)

        return final_text_vector, final_image_vector, final_fused_vector, logits
        
  

In [5]:

class MultimodalSentimentModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.text_model = ALBERTBiLSTMFeatureExtractor()
        self.image_model = DenseNetCBAMFeatureExtractor()
        self.fusion = GatedCrossAttentionNetwork()

    def forward(self, image_tensor, text_input_str):
        # Extract features
        text_features = self.text_model(text_input_str)        # [batch_size, 512]
        image_features = self.image_model(image_tensor)        # [batch_size, 512]

        # Fusion and classification
        _, _, _, logits = self.fusion(text_features, image_features)
        return logits


# Initialize Model
model = MultimodalSentimentModel(dropout_rate=0.3)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)  

# Training Loop
num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for image_features, text_features, labels in train_loader:
        # Get outputs
        final_text_vector, final_image_vector, final_fused_vector, logits = model(text_features, image_features)

        # Compute loss using classification logits
        loss = criterion(logits, labels)
        total_loss += loss.item()
        
        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_loss:.4f}")



# Evaluation on test data
model.eval()
all_labels = []
all_preds = []

with torch.no_grad():
    for image_features, text_features, labels in test_loader:
        # Unpack all outputs
        final_text_vector, final_image_vector, final_fused_vector, logits = model(text_features, image_features)

        # Use logits for prediction
        _, preds = torch.max(logits, 1)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

# Calculate accuracy, precision, and confusion matrix
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average="weighted")
conf_matrix = confusion_matrix(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average="weighted")

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test F1 Score: {f1:.4f}")
print("Confusion Matrix:\n", conf_matrix)

# Sample prediction
sample_image_features, sample_text_features, sample_label = test_data[0]

# Unsqueeze to match batch format
with torch.no_grad():
    final_text_vector, final_image_vector, final_fused_vector, sample_output = model(
        sample_text_features.unsqueeze(0), sample_image_features.unsqueeze(0)
    )

# Get predicted label
_, sample_pred = torch.max(sample_output, 1)
print(f"Sample True Label: {sample_label}, Predicted Label: {sample_pred.item()}")

torch.save(model.state_dict(), "my_multimodal_sentement_classifier_model.pth")

/tmp/ipykernel_30/2003254916.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  all_features_512 = torch.load(feature_path)
Epoch [1/20], Train Loss: 0.9802
Epoch [2/20], 